# 🤖 LangChain — AI Agents
## Python Ecosystem Tutorial Series — Module 14 of 18

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

| | |
|---|---|
| **Library** | 🤖 LangChain |
| **Domain** | AI Agents |
| **Dataset** | Drug safety agent |
| **Module** | 14 of 18 |

**What you will learn:**

1. What LangChain is and why it exists
2. Core concepts and data structures
3. Hands-on code with real data
4. Visualisations and interpretation
5. When to use it and alternatives

```bash
# Install required libraries
pip install langchain langchain-openai
```

## Quick Reference Card

| Code | What it does |
|------|--------------|
| `@tool` | Define a tool |
| `ChatOpenAI()` | LLM instance |
| `AgentExecutor.invoke()` | Run agent |
| `ChatPromptTemplate` | Structure the prompt |
| `tool_calls` | LLM-requested tool calls |

# 14. 🤖 LangChain — AI Agents
> **Python + LangChain = AI Agents**

LangChain orchestrates LLMs with tools, memory, and data. Build chatbots,
RAG (Retrieval-Augmented Generation) systems, and autonomous agents.

**Key concepts:** LLM chains, prompt templates, tools, agents, RAG, memory, vector stores

In [ ]:
# ── LangChain AI Agent for scientific literature ─────────────────────────────
try:
    from langchain_openai import ChatOpenAI
    from langchain.agents import create_tool_calling_agent, AgentExecutor
    from langchain_core.tools import tool
    from langchain_core.prompts import ChatPromptTemplate
    LC_OK = True
except ImportError:
    LC_OK = False
    print("pip install langchain langchain-openai")

import os
from openai import OpenAI
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── The core concept: Tools + LLM = Agent ─────────────────────────────────────
print("LangChain Architecture:")
print("""
  User Question
       │
       ▼
   LLM (GPT-4o)
       │ decides which tool(s) to call
       ▼
  ┌────────────────────────────────────┐
  │  Tool 1: search_pubmed             │
  │  Tool 2: lookup_compound_property  │
  │  Tool 3: run_qsar_model            │
  │  Tool 4: fetch_clinical_trial      │
  └────────────────────────────────────┘
       │ tool results returned
       ▼
   LLM synthesises final answer
""")

In [ ]:
# ── Build a working agent (raw OpenAI if LangChain not installed) ─────────────
AGENT_OK = bool(os.getenv("OPENAI_API_KEY",""))

# ── Tool functions ───────────────────────────────────────────────────────────
DRUG_DB = {
    "aspirin":       {"mw":180.2, "logp":1.19, "category":"NSAID",         "status":"approved", "ld50_mg_kg":200},
    "caffeine":      {"mw":194.2, "logp":-0.07,"category":"stimulant",      "status":"GRAS",     "ld50_mg_kg":192},
    "ibuprofen":     {"mw":206.3, "logp":3.97, "category":"NSAID",         "status":"approved", "ld50_mg_kg":636},
    "paracetamol":   {"mw":151.2, "logp":0.46, "category":"analgesic",     "status":"approved", "ld50_mg_kg":338},
    "warfarin":      {"mw":308.3, "logp":2.70, "category":"anticoagulant", "status":"approved", "ld50_mg_kg":3},
    "atrazine":      {"mw":215.7, "logp":2.61, "category":"herbicide",     "status":"regulated","ld50_mg_kg":1869},
    "bisphenol_a":   {"mw":228.3, "logp":3.32, "category":"industrial",    "status":"restricted","ld50_mg_kg":3250},
    "pfoa":          {"mw":414.1, "logp":4.81, "category":"PFAS",          "status":"banned",   "ld50_mg_kg":430},
}

PAPERS_DB = {
    "aspirin": ["Wang et al 2024: Aspirin reduces colorectal cancer risk by 19%",
                "Nissen et al 2023: Low-dose aspirin: meta-analysis of 45 trials"],
    "pfoa":    ["EPA 2024: PFOA MCL set at 4 ng/L in drinking water",
                "Chang et al 2023: PFOA associated with thyroid cancer (OR=1.58)"],
    "caffeine":["Nehlig 2022: Caffeine mechanisms in adenosine receptor antagonism",
                "Kim et al 2023: 400 mg/day caffeine safe for healthy adults"],
}

def tool_drug_lookup(drug_name: str) -> dict:
    """Look up physicochemical properties of a drug."""
    key = drug_name.lower().replace(" ","_")
    return DRUG_DB.get(key, {"error": f"Drug {drug_name!r} not in database"})

def tool_search_literature(drug_name: str) -> dict:
    """Search recent papers about a compound."""
    key = drug_name.lower()
    papers = PAPERS_DB.get(key, [f"No recent papers found for {drug_name}"])
    return {"drug": drug_name, "papers": papers, "count": len(papers)}

def tool_lipinski_check(mw: float, logp: float, hba: int, hbd: int) -> dict:
    """Check Lipinski Rule of 5 for oral bioavailability."""
    rules = {"MW<=500": mw<=500, "logP<=5": logp<=5, "HBA<=10": hba<=10, "HBD<=5": hbd<=5}
    pass_count = sum(rules.values())
    return {"rules": rules, "pass_count": pass_count,
            "verdict": "PASS" if pass_count>=4 else "FAIL",
            "oral_bioavailability": "Good" if pass_count>=4 else "Poor"}

def tool_ghs_classify(ld50_mg_kg: float) -> dict:
    """Classify acute oral toxicity by GHS category."""
    if ld50_mg_kg <=    5: cat, signal = 1, "Danger (Fatal)"
    elif ld50_mg_kg <=  50: cat, signal = 2, "Danger"
    elif ld50_mg_kg <= 300: cat, signal = 3, "Danger"
    elif ld50_mg_kg <=2000: cat, signal = 4, "Warning"
    else:                   cat, signal = 5, "Low hazard"
    return {"ghs_category": cat, "signal_word": signal, "ld50": ld50_mg_kg}

TOOL_REGISTRY = {"drug_lookup":tool_drug_lookup, "search_literature":tool_search_literature,
                  "lipinski_check":tool_lipinski_check, "ghs_classify":tool_ghs_classify}

print("Tools ready:", list(TOOL_REGISTRY.keys()))

In [ ]:
# ── Agent loop ───────────────────────────────────────────────────────────────
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY",""))

TOOLS_SCHEMA = [
    {"type":"function","function":{"name":"drug_lookup",
     "description":"Look up physicochemical properties (MW, logP, category, LD50) of a drug.",
     "parameters":{"type":"object","properties":{"drug_name":{"type":"string"}},"required":["drug_name"]}}},
    {"type":"function","function":{"name":"search_literature",
     "description":"Find recent scientific papers about a compound or drug.",
     "parameters":{"type":"object","properties":{"drug_name":{"type":"string"}},"required":["drug_name"]}}},
    {"type":"function","function":{"name":"lipinski_check",
     "description":"Check drug-likeness using Lipinski Rule of 5.",
     "parameters":{"type":"object","properties":{
         "mw":{"type":"number"},"logp":{"type":"number"},
         "hba":{"type":"integer"},"hbd":{"type":"integer"}},"required":["mw","logp","hba","hbd"]}}},
    {"type":"function","function":{"name":"ghs_classify",
     "description":"Classify acute oral toxicity into GHS category using LD50.",
     "parameters":{"type":"object","properties":{"ld50_mg_kg":{"type":"number"}},"required":["ld50_mg_kg"]}}},
]

SYSTEM = """You are a pharmaceutical AI assistant. For each compound: call ALL relevant tools,
then give a concise 3-sentence safety + property summary citing the evidence."""

def run_langchain_agent(question: str):
    if not AGENT_OK:
        # Offline demo for aspirin
        drug = tool_drug_lookup("aspirin")
        papers = tool_search_literature("aspirin")
        lip = tool_lipinski_check(drug["mw"],drug["logp"],4,2)
        ghs = tool_ghs_classify(drug["ld50_mg_kg"])
        return (f"[Demo - no API key] Aspirin: MW={drug['mw']}Da, logP={drug['logp']}, "
                f"GHS Cat {ghs['ghs_category']} ({ghs['signal_word']}), "
                f"Lipinski: {lip['verdict']}. Papers: {papers['papers'][0][:60]}...")
    msgs = [{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    for _ in range(8):
        r = client.chat.completions.create(model="gpt-4o",messages=msgs,tools=TOOLS_SCHEMA,tool_choice="auto")
        c = r.choices[0]; msgs.append(c.message)
        if c.finish_reason=="stop": return c.message.content
        for tc in c.message.tool_calls:
            res = TOOL_REGISTRY[tc.function.name](**json.loads(tc.function.arguments))
            msgs.append({"role":"tool","tool_call_id":tc.id,"content":json.dumps(res)})
    return "max iterations"

# Demo queries
for q in [
    "Assess aspirin: its drug-likeness, toxicity class, and what recent research says.",
    "Is PFOA dangerous? What does the literature say?"
]:
    print(f"Q: {q}")
    print(f"A: {run_langchain_agent(q)}")
    print()

# ── RAG concept diagram ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0,14); ax.set_ylim(0,5); ax.axis("off")
ax.set_facecolor("#F8F9FA")

steps = [
    (1.0,  "1\nUser\nQuestion", "#3498DB"),
    (3.5,  "2\nLLM decides\nwhich tools", "#8E44AD"),
    (6.5,  "3\nTool calls\n(DB, search, calc)", "#27AE60"),
    (9.5,  "4\nResults\nreturned", "#E67E22"),
    (12.5, "5\nFinal answer\n(cited)", "#E74C3C"),
]
for x, label, col in steps:
    ax.add_patch(mpatches.FancyBboxPatch((x-0.9,1.5),1.8,2.0,
        boxstyle="round,pad=0.12",facecolor=col,edgecolor="white",alpha=0.85,lw=2))
    ax.text(x, 2.5, label, ha="center", va="center",
             color="white", fontweight="bold", fontsize=9)

for i in range(len(steps)-1):
    ax.annotate("", xy=(steps[i+1][0]-0.9,2.5), xytext=(steps[i][0]+0.9,2.5),
                 arrowprops=dict(arrowstyle="->",color="#1F2937",lw=2))

tools = ["drug_lookup","search_literature","lipinski_check","ghs_classify"]
for i, t in enumerate(tools):
    ax.text(6.5, 1.2 - i*0.25, f"• {t}()", color="#27AE60", fontsize=8)

ax.text(7, 4.6, "LangChain Agent Loop", ha="center", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("langchain_agent.png", dpi=120, bbox_inches="tight")
plt.show()

## Deep Dive: LangChain

### The Problem with LLMs Alone
GPT-4 knows a lot but: has a knowledge cutoff, cannot access your private database, cannot run calculations, and hallucinates when uncertain. LangChain's tool-calling solves this by giving the LLM external capabilities.

### How Tool-Calling Works
The LLM receives a list of tool descriptions. When it determines a tool would help answer the question, it outputs a tool_call object (not text). Your code executes the actual function and returns the result. The LLM then synthesises the result into a natural language answer.

### The ReAct Loop in Practice
```
User: "Is aspirin safe? What does the literature say?"

Thought: I need drug properties and literature
Action: drug_lookup("aspirin") -> {mw:180, logp:1.19, ld50:200}
Action: search_literature("aspirin") -> ["Wang 2024: reduces cancer risk...", ...]
Action: ghs_classify(200) -> {category:4, signal_word:"Warning"}

Final Answer: Aspirin has GHS Category 4 (Warning) acute toxicity,
meaning it's relatively safe. Recent literature (Wang et al. 2024)
shows it reduces colorectal cancer risk by 19%...
```

### RAG Architecture
Vector database stores document embeddings. At query time, the user's question is embedded, the database returns the most similar document chunks, and these chunks are included in the LLM prompt. The LLM sees only the relevant context, not the entire corpus.

### Choosing Your AI Framework
LangChain: largest ecosystem, most integrations.
LlamaIndex: specialises in document indexing and RAG.
AutoGen: multi-agent conversations.
Raw OpenAI SDK: simplest when you don't need the extra abstractions.


## ✅ Key Takeaways — 🤖 LangChain

1. LangChain turns LLMs from 'know-it-all' to 'do-it-all' by adding tools
2. Tool descriptions must be precise — the LLM reasons about which tool to call
3. RAG prevents hallucination by grounding answers in retrieved documents
4. The ReAct pattern (Reason + Act) is the foundation of most modern AI agents

---
*Next: Continue to Module 15 of 18 in the Python Ecosystem Tutorial Series*  
*Portfolio: [hgoelgithub.github.io](https://hgoelgithub.github.io)*